In [1]:
# --- Cell 1: Sliding-window + text handling utilities (lossless, no rewrites) ---

import json
from typing import List, Tuple, Dict, Any
from pathlib import Path

def split_paragraphs_lossless(text: str) -> List[str]:
    if text is None:
        return []
    if not isinstance(text, str):
        text = str(text)

    lines = text.splitlines()
    paras: List[str] = []
    buf: List[str] = []

    for line in lines:
        if line.strip() == "":
            if buf:
                paras.append("\n".join(buf))
                buf = []
        else:
            buf.append(line)
    if buf:
        paras.append("\n".join(buf))

    return paras


def build_sliding_windows(
    paragraphs: List[str],
    max_chars: int = 3200,
    overlap_paras: int = 1
) -> List[Tuple[int, List[str]]]:

    if max_chars < 200:
        raise ValueError("max_chars is too small; use >= 200.")
    if overlap_paras < 0:
        raise ValueError("overlap_paras must be >= 0.")

    windows: List[Tuple[int, List[str]]] = []
    start = 0
    n = len(paragraphs)

    while start < n:
        total = 0
        win: List[str] = []
        i = start

        while i < n:
            p = paragraphs[i]
            if win and (total + 2 + len(p)) > max_chars:
                break
            if not win and len(p) > max_chars:
                win = [p]
                i += 1
                break
            if win:
                total += 2
            total += len(p)
            win.append(p)
            i += 1

        windows.append((start, win))

        if i >= n:
            break

        start = max(i - overlap_paras, start + 1)

    return windows


def build_window_prompt(window_paragraphs: List[str], start_index: int) -> str:
    """
    Prompt is now WINDOW-AWARE.
    Model is explicitly forbidden from inventing paragraph ids.
    """

    allowed_ids = [start_index + i + 1 for i in range(len(window_paragraphs))]

    joined = "\n\n".join(
        f"[P{pid}]\n{window_paragraphs[i]}"
        for i, pid in enumerate(allowed_ids)
    )

    schema = {
        "paragraphs": [
            {
                "p": allowed_ids[0],
                "claims": [],
                "evidence_mentioned": [],
                "evidence_to_request": [],
                "evidence_to_locate_own": [],
                "confidence": "high"
            }
        ]
    }

    return f"""
You will be given a set of paragraphs. Your job is to output structured JSON for ONLY those paragraphs.

STRICT RULES (NON-NEGOTIABLE):
1) DO NOT change, rewrite, paraphrase, “clean”, or correct the input text.
2) Do NOT output the paragraph text itself. Do NOT reproduce or restate any of the input text.
3) Identify paragraphs ONLY by their paragraph number `p`.
4) ALLOWED paragraph IDs for this window are EXACTLY: {allowed_ids}. Do NOT invent or renumber paragraphs.
5) Put any interpretation only in `claims` (and set `confidence`).
6) `evidence_mentioned` must list ONLY evidence explicitly referenced in the text.
7) If evidence is not explicitly referenced, use `evidence_to_request` or `evidence_to_locate_own` (generic phrasing only).
8) Output JSON ONLY. No markdown. No commentary. No extra keys.

OUTPUT JSON SCHEMA (match keys exactly):
{json.dumps(schema, ensure_ascii=False)}

Paragraphs:
<<<
{joined}
>>>
""".strip()


def build_section_title_summary_prompt(paragraph_objs: List[Dict[str, Any]]) -> str:
    claims_block = "\n".join(
        f"P{p['p']}: {p.get('claims', [])}"
        for p in sorted(paragraph_objs, key=lambda x: x["p"])
    )

    schema = {"section_title": "", "section_summary": ""}

    return f"""
You will be given CLAIMS (not the original text). Produce a section title and summary.

STRICT RULES:
1) Base your output ONLY on the claims below. Do NOT add new facts.
2) Title rules: 3–7 words, nouns only, no punctuation.
3) Output JSON ONLY, matching schema exactly. No extra keys.

OUTPUT JSON SCHEMA:
{json.dumps(schema, ensure_ascii=False)}

CLAIMS:
<<<
{claims_block}
>>>
""".strip()


In [3]:
# --- Cell 2: LLM calls + CSV -> JSONL runner (Ollama CLI) ---

import pandas as pd
import subprocess
import hashlib
from tqdm.auto import tqdm


# Paths (you said both CSV + output should be in the same folder)
BASE_DIR = Path("/home/hello/Projects/Statements/input")
IN_CSV   = BASE_DIR / "Leonardo_WS.csv"
OUT_JL   = BASE_DIR / "WS_LFMM_vs_BBG.jsonl"

# Config
TEXT_COL       = "text_verbatim"   # <-- set this to the column you want the LLM to read row-by-row
MODEL          = "mistral-small3.2:latest"  # <-- your local model name in Ollama
MAX_CHARS      = 3200
OVERLAP_PARAS  = 1
MAKE_TITLE_SUMMARY = True

import json
import hashlib
import requests
from typing import Dict, Any

OLLAMA_URL = "http://127.0.0.1:11434/api/generate"

def canonicalize_verbatim(s: str) -> str:
    if s is None:
        return ""
    if not isinstance(s, str):
        s = str(s)

    # normalize newlines
    s = s.replace("\r\n", "\n").replace("\r", "\n")

    # strip ONLY outer whitespace (models often add a trailing newline)
    s = s.strip()

    # normalize common smart quotes to straight quotes (optional but often needed)
    s = (s.replace("“", '"').replace("”", '"')
           .replace("‘", "'").replace("’", "'"))

    # normalize non-breaking space to normal space
    s = s.replace("\u00A0", " ")

    # remove zero-width chars that sometimes appear in copied text
    s = s.replace("\u200b", "").replace("\ufeff", "")

    return s


def strip_json_fence(raw: str) -> str:
    """
    If the model returns fenced JSON like ```json ... ```, strip the fences.
    Otherwise return raw unchanged.
    """
    if raw is None:
        return ""
    s = raw.strip()

    if s.startswith("```"):
        # remove first fence line
        first_nl = s.find("\n")
        if first_nl != -1:
            s = s[first_nl + 1:].strip()
        # remove trailing fence
        if s.endswith("```"):
            s = s[:-3].strip()

    return s


def ollama_run(model: str, prompt: str, timeout: int = 600) -> str:
    """
    Calls *local* Ollama via HTTP API (localhost). Returns raw text output.
    Streaming=True to avoid hangs and handle large outputs robustly.
    """
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": True,
    }

    chunks = []
    with requests.post(OLLAMA_URL, json=payload, stream=True, timeout=timeout) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            if not line:
                continue
            obj = json.loads(line.decode("utf-8"))
            if "response" in obj:
                chunks.append(obj["response"])
            if obj.get("done"):
                break

    return "".join(chunks).strip()

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def parse_json_strict(raw: str) -> dict:
    raw2 = strip_json_fence(raw)
    try:
        return json.loads(raw2)
    except json.JSONDecodeError as e:
        raise ValueError(
            f"Model did not return valid JSON. Error: {e}\n"
            f"--- Raw output (first 1200 chars) ---\n{raw[:1200]}"
        )


def process_section_text(section_text: str) -> Dict[str, Any]:
    """
    Sliding-window extraction for one section (one CSV row).
    Returns a single JSON object with section_title/summary + paragraphs.
    Enforces verbatim by hashing original paragraph text and comparing with returned `text_verbatim`.
    """
    paras = split_paragraphs_lossless(section_text)
    if not paras:
        return {"section_title": "", "section_summary": "", "paragraphs": []}

    windows = build_sliding_windows(paras, max_chars=MAX_CHARS, overlap_paras=OVERLAP_PARAS)

    truth_hash = {i + 1: sha256_text(canonicalize_verbatim(paras[i])) for i in range(len(paras))}
    collected: Dict[int, Dict[str, Any]] = {}

    for start_idx, win_paras in windows:
        prompt = build_window_prompt(win_paras, start_idx)
        raw = ollama_run(MODEL, prompt)
        obj = parse_json_strict(raw)

        if "paragraphs" not in obj or not isinstance(obj["paragraphs"], list):
            raise ValueError("JSON missing required key `paragraphs` as a list.")

        for p in obj["paragraphs"]:
            pid = p.get("p", None)

            # Validate pid
            if not isinstance(pid, int) or pid < 1 or pid > len(paras):
                raise ValueError(f"Invalid paragraph id in output: {pid}")

            # Inject ground-truth verbatim text (model never supplies it)
            p["text_verbatim"] = paras[pid - 1]

            # Optional: ensure required analysis keys exist (keeps schema consistent)
            p.setdefault("claims", [])
            p.setdefault("evidence_mentioned", [])
            p.setdefault("evidence_to_request", [])
            p.setdefault("evidence_to_locate_own", [])
            p.setdefault("confidence", "high")

            # First-win policy in overlaps (deterministic)
            if pid not in collected:
                collected[pid] = p

    paragraph_list = [collected[k] for k in sorted(collected.keys())]

    section_title = ""
    section_summary = ""

    if MAKE_TITLE_SUMMARY and paragraph_list:
        prompt2 = build_section_title_summary_prompt(paragraph_list)
        raw2 = ollama_run(MODEL, prompt2)
        obj2 = parse_json_strict(raw2)
        section_title = obj2.get("section_title", "") or ""
        section_summary = obj2.get("section_summary", "") or ""

    return {
        "section_title": section_title,
        "section_summary": section_summary,
        "paragraphs": paragraph_list
    }


# ---- Run: CSV -> JSONL ----

if not IN_CSV.exists():
    raise FileNotFoundError(f"CSV not found: {IN_CSV}")

df = pd.read_csv(IN_CSV, dtype=str, keep_default_na=False)

if TEXT_COL not in df.columns:
    raise KeyError(f"Column '{TEXT_COL}' not found. Available columns: {list(df.columns)}")

written = 0
rows = df[TEXT_COL].tolist()

with open(OUT_JL, "w", encoding="utf-8") as f:
    for row_idx, section_text in tqdm(enumerate(rows), total=len(rows), desc="WS -> JSONL"):
        section_text = (section_text or "").strip()
        if not section_text:
            continue

        out_obj = process_section_text(section_text)
        out_obj["_source_row"] = row_idx
        f.write(json.dumps(out_obj, ensure_ascii=False) + "\n")
        written += 1

print(f"Wrote {written} JSON objects to: {OUT_JL}")
# --- End of script ---

In [4]:
import pandas as pd

path = "/home/hello/Projects/Statements/input/WS_LFMM_vs_BBG.jsonl"

df = pd.read_json(path, lines=True)

df.head()


rows = []

for _, r in df.iterrows():
    section_title = r.get("section_title", "")
    section_summary = r.get("section_summary", "")
    source_row = r.get("_source_row", None)

    for p in r.get("paragraphs", []):
        rows.append({
            "source_row": source_row,
            "section_title": section_title,
            "section_summary": section_summary,
            "p": p.get("p"),
            "text_verbatim": p.get("text_verbatim"),
            "claims": p.get("claims"),
            "evidence_mentioned": p.get("evidence_mentioned"),
            "evidence_to_request": p.get("evidence_to_request"),
            "evidence_to_locate_own": p.get("evidence_to_locate_own"),
            "confidence": p.get("confidence"),
        })

pdf = pd.DataFrame(rows)
pdf.to_csv("/home/hello/Projects/Statements/input/WS_LFMM_vs_BBG_expanded.csv", index=False)


# Witness Statement PDF Generator

This script converts a structured CSV version of the Witness Statement into a
formatted PDF using ReportLab.

It supports **two output modes**, controlled by a single boolean flag.

---

## 🔧 Configuration

```python
ANALYSIS = True  # or False



In [1]:
import json
import re
from pathlib import Path

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.pdfgen import canvas
from reportlab.lib.utils import simpleSplit

# ========================
# CONFIG
# ========================
JSON_PATH = "/home/hello/Projects/Statements/input/WS_LFMM_vs_BBG.jsonl"  # .jsonl or .json
OUT_PDF   = "/home/hello/Projects/Statements/input/Leonardo_WS.pdf"

ANALYSIS = True  # True = analysis pack; False = clean ET WS (number every paragraph)

# ---- PDF layout ----
PAGE_W, PAGE_H = A4
LEFT = 2.0 * cm
RIGHT = 2.0 * cm
TOP = 2.0 * cm
BOTTOM = 2.0 * cm
MAX_W = PAGE_W - LEFT - RIGHT

FONT = "Times-Roman"
FONT_B = "Times-Bold"
SIZE = 11
LEADING = 14  # line height

# ========================
# IO
# ========================
def load_json_any(path: str):
    """
    Loads either:
      - JSONL: one object per line
      - JSON: a list of objects or a single object
    Returns a list[dict].
    """
    p = Path(path)
    txt = p.read_text(encoding="utf-8").strip()
    if not txt:
        return []

    if p.suffix.lower() == ".jsonl":
        out = []
        for line in txt.splitlines():
            line = line.strip()
            if not line:
                continue
            out.append(json.loads(line))
        return out

    # .json
    obj = json.loads(txt)
    if isinstance(obj, list):
        return obj
    return [obj]

# ========================
# TEXT HELPERS
# ========================
def _norm_text(x) -> str:
    if x is None:
        return ""
    s = str(x)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    return s

def strip_leading_numbering(s: str) -> str:
    """
    Remove existing numbering like:
      '1. ', '1.1 ', '2.3.4 ', '(1) ', '(1.2) ', '154. '
    """
    s = (s or "").strip()
    if not s:
        return ""
    # Remove leading patterns like 154. or 1.1. or (12)
    s = re.sub(r"^\(?\s*\d+(?:\.\d+)*\s*\)?\s*[.)]?\s+", "", s)
    return s.strip()

def extract_original_paragraph_no(s: str):
    """
    Extract leading WS-style paragraph numbers like:
      '13. ...', '154. ...'
    Returns int or None.
    """
    if not s:
        return None
    m = re.match(r"\s*(\d+)\s*[.)]\s+", s)
    if m:
        try:
            return int(m.group(1))
        except ValueError:
            return None
    return None

def split_ws_paragraphs(text: str) -> list[str]:
    """
    Clean WS split:
    - If the text has blank-line paragraphing, respect it.
    - Otherwise, treat each non-empty line as a paragraph.
    """
    s = _norm_text(text).strip()
    if not s:
        return []

    # If blank lines exist, use them as paragraph breaks
    if re.search(r"\n\s*\n", s):
        parts = re.split(r"\n\s*\n+", s)
        paras = []
        for p in parts:
            p = p.strip()
            if not p:
                continue
            # collapse internal newlines within that paragraph
            p = re.sub(r"\s*\n\s*", " ", p).strip()
            paras.append(p)
        return paras

    # Otherwise: each non-empty line is a paragraph
    lines = [ln.strip() for ln in s.splitlines() if ln.strip()]
    return lines

# ========================
# DRAW HELPERS
# ========================
def _ensure_room(c, y, needed_lines=2):
    if y < BOTTOM + LEADING * needed_lines:
        c.showPage()
        return PAGE_H - TOP
    return y

def draw_wrapped(c, text, x, y, max_w, font=FONT, size=SIZE):
    text = _norm_text(text)
    c.setFont(font, size)
    lines = simpleSplit(text, font, size, max_w)
    for line in lines:
        if y < BOTTOM + LEADING:
            c.showPage()
            y = PAGE_H - TOP
            c.setFont(font, size)
        c.drawString(x, y, line)
        y -= LEADING
    return y

def draw_label_value(c, label, value, y):
    y = _ensure_room(c, y, needed_lines=3)
    c.setFont(FONT_B, 12)
    c.drawString(LEFT, y, label)
    y -= LEADING
    y = draw_wrapped(c, value, LEFT, y, MAX_W, font=FONT, size=SIZE)
    y -= LEADING * 0.6
    return y

def draw_heading(c, text, y):
    y = _ensure_room(c, y, needed_lines=3)
    c.setFont(FONT_B, 13)
    y = draw_wrapped(c, text, LEFT, y, MAX_W, font=FONT_B, size=13)
    y -= LEADING * 0.2
    return y

def draw_numbered_paragraph(c, number: int, para_text: str, y: float):
    """
    ET-style numbered paragraph with FIXED hanging indent so alignment does not shift
    as paragraph numbers grow (e.g. 9. vs 154.).
    """
    para_text = _norm_text(para_text).strip()
    if not para_text:
        return y

    c.setFont(FONT, SIZE)

    # Fixed indent: reserve space for "9999.  " (adjust if you expect >9999 paragraphs)
    fixed_prefix = "9999.  "
    indent_w = c.stringWidth(fixed_prefix, FONT, SIZE)

    prefix = f"{number}."
    gap = "  "

    # Ensure room to start paragraph
    if y < BOTTOM + LEADING * 3:
        c.showPage()
        y = PAGE_H - TOP
        c.setFont(FONT, SIZE)

    # Wrap using fixed indent width
    lines = simpleSplit(para_text, FONT, SIZE, MAX_W - indent_w)
    if not lines:
        return y

    # Right-align number within the fixed indent column
    prefix_text = prefix + gap
    prefix_w = c.stringWidth(prefix_text, FONT, SIZE)
    x_num = LEFT + (indent_w - prefix_w)

    c.drawString(x_num, y, prefix_text)
    c.drawString(LEFT + indent_w, y, lines[0])
    y -= LEADING

    for line in lines[1:]:
        if y < BOTTOM + LEADING:
            c.showPage()
            y = PAGE_H - TOP
            c.setFont(FONT, SIZE)
        c.drawString(LEFT + indent_w, y, line)
        y -= LEADING

    y -= LEADING * 0.6
    return y

# ========================
# MAIN
# ========================
sections = load_json_any(JSON_PATH)

c = canvas.Canvas(OUT_PDF, pagesize=A4)
y = PAGE_H - TOP

para_counter = 1

for s in sections:
    section_title = s.get("section_title", "")
    section_summary = s.get("section_summary", "")
    paragraphs = s.get("paragraphs", []) or []

    if ANALYSIS:
        # --- Analysis pack: human + QA friendly
        if section_title:
            y = draw_heading(c, section_title, y)

        if section_summary:
            y = draw_label_value(c, "Summary", section_summary, y)

        for p in paragraphs:
            pnum = p.get("p", "")
            y = draw_heading(c, f"Paragraph block {pnum}".strip(), y)

            # Claims (robust to dicts or strings; skip empty text)
            claims = p.get("claims", []) or []
            clean_claims = []
            for cl in claims:
                if isinstance(cl, dict):
                    txt = (cl.get("text") or "").strip()
                    conf = (cl.get("confidence") or "").strip()
                else:
                    txt = str(cl).strip()
                    conf = ""
                if txt:
                    clean_claims.append((txt, conf))

            if clean_claims:
                y = draw_label_value(c, "Claims", "", y)
                for txt, conf in clean_claims:
                    line = f"- {txt}" + (f"  [{conf}]" if conf else "")
                    y = draw_wrapped(c, line, LEFT + 0.5 * cm, y, MAX_W - 0.5 * cm)
                y -= LEADING * 0.6

            # Evidence lists
            for lab, key in [
                ("Evidence mentioned", "evidence_mentioned"),
                ("Evidence to request", "evidence_to_request"),
                ("Evidence to locate own", "evidence_to_locate_own"),
            ]:
                items = p.get(key, []) or []
                if items:
                    y = draw_label_value(c, lab, "", y)
                    for it in items:
                        y = draw_wrapped(c, f"- {it}", LEFT + 0.5 * cm, y, MAX_W - 0.5 * cm)
                    y -= LEADING * 0.6

            # Verbatim
            text_verbatim = p.get("text_verbatim", "")
            if text_verbatim:
                y = draw_label_value(c, "Text verbatim", text_verbatim, y)

        # page break between sections (analysis only)
        c.showPage()
        y = PAGE_H - TOP


if not ANALYSIS:
    # -------- CLEAN WS MODE (ORDER + FORMAT LOCKED) --------
    flat = []
    seq = 0

    # Pass 1: flatten all paragraphs from all sections
    for s in sections:
        paragraphs = s.get("paragraphs", []) or []
        for p in paragraphs:
            text_verbatim = p.get("text_verbatim", "")
            paras = split_ws_paragraphs(text_verbatim)

            for para in paras:
                raw = (para or "").strip()
                if not raw:
                    continue
                orig_no = extract_original_paragraph_no(raw)
                clean_text = strip_leading_numbering(raw)

                if not clean_text:
                    continue

                flat.append({
                    "orig_no": orig_no,
                    "seq": seq,
                    "text": clean_text,
                })
                seq += 1

    # Pass 2: sort by original paragraph number if present, else by appearance order
    flat.sort(key=lambda x: (x["orig_no"] is None, x["orig_no"] if x["orig_no"] is not None else x["seq"]))

    # Pass 3: render as 1..n (fixed indent)
    for item in flat:
        y = draw_numbered_paragraph(c, para_counter, item["text"], y)
        para_counter += 1

c.save()
print("Wrote PDF:", OUT_PDF)


Wrote PDF: /home/hello/Projects/Statements/input/Leonardo_WS.pdf
